
# Afegir soroll a fingerprints ECFP4 en format espars

Aquest notebook:

- Llegeix un CSV amb una columna de fingerprints ECFP4 en format espars (llista d'índexs on el bit és 1).
- Assumeix que el fingerprint té **2048 bits**.
- Introdueix un error de:
  - `noise_0` % en els **zeros** del fingerprint (bits 0 → 1 amb aquesta probabilitat).
  - `noise_1` % en els **uns** del fingerprint (bits 1 → 0 amb aquesta probabilitat).
- **Només funciona** per fingerprints ECFP4 representats com una llista d'índexs (0-based) on hi ha bits a 1.
- Desa un nou CSV amb la **mateixa estructura** però amb la columna de fingerprints modificada.


## Imports

In [1]:
import os
import pandas as pd
import numpy as np

## Inputs (part a editar)

Arrel del projecte

In [2]:
#os.chdir("/export/home/ddiestre/MolForge_Testing/data/MolForge_input")
#os.chdir("/mnt/c/Users/david/Desktop/Uni/MolForge_Testing/data/MolForge_input")
os.chdir("/mnt/d/MolForge_Testing/data/MolForge_input")

Fitxer de fingerprints sense soroll

In [3]:
#input_path = "MolForge_MFinput_2000_ECFP4.csv"
input_path = "CoCoGraph_MFinput_2000_novel.csv"
#input_path = "CoCoGraph_MFinput_2000_lt70atoms.csv"
#input_path = "PubChem_MFinput_2000_CID-SMILES-filtered-lt70.csv"

Percentatges de soroll

In [4]:
noise_0 = 10    # % d'errors en bits que són 0 (0 -> 1)
noise_1 = 10    # % d'errors en bits que són 1 (1 -> 0)

Columna de fingerprints

In [5]:
# Nom de la columna amb el fingerprint espars ECFP4
fp_in_col = "fingerprints_input_ECFP4"

# Nombre de bits del fingerprint ECFP4
n_bits = 2048

Reproduïbilitat

In [6]:
random_seed = 42
rng = np.random.default_rng(random_seed)

In [7]:
# Separar nom i extensió
base_name = os.path.basename(input_path)   # "MolForge_MFinput_2000_ECFP4.csv"
name, ext = os.path.splitext(base_name)    # ("MolForge_MFinput_2000_ECFP4", ".csv")

# Crear carpeta de sortida: "{name}_noise"
output_dir = f"{name}_noise"
os.makedirs(output_dir, exist_ok=True)

# Convertir noise a format sense punt per no guardar el nom del output amb punts
noise0_str = str(noise_0).replace(".", "_")
noise1_str = str(noise_1).replace(".", "_")

# Nom del fitxer de sortida (dins la carpeta)
output_filename = f"{name}_noise0_{noise0_str}_noise1_{noise1_str}{ext}"
output_path = os.path.join(output_dir, output_filename)


## 0. Funcions auxiliars

In [8]:
def parse_sparse_fp(fp_str: str) -> set:
    """
    Converteix una cadena amb índexs separats per espais en un set d'índexs (ints).
    Exemple: "80 222 473" -> {80, 222, 473}

    Es considera que:
    - Tenim un fingerprint de longitud n_bits.
    - Els índexs són 0-based i sempre < n_bits.
    """
    if pd.isna(fp_str):
        return set()
    s = str(fp_str).strip()
    if not s:
        return set()
    return {int(x) for x in s.split()}


def fp_set_to_sparse_str(indices_set: set) -> str:
    """
    Converteix un set d'índexs (bits a 1) en una cadena ordenada amb espais.
    Exemple: {222, 80, 473} -> "80 222 473"
    """
    return " ".join(str(i) for i in sorted(indices_set))


def apply_noise_to_fp_sparse(indices_str: str,
                             n_bits: int,
                             noise_0: float,
                             noise_1: float,
                             rng: np.random.Generator) -> str:
    """
    Aplica soroll a un fingerprint ECFP4 en format espars.

    - indices_str: cadena amb índexs a 1 (ex: "80 222 473")
    - n_bits: longitud total del fingerprint (ex: 2048)
    - noise_0: % d'errors en zeros (0 -> 1)
    - noise_1: % d'errors en uns (1 -> 0)
    - rng: generador de nombres aleatoris (np.random.Generator)

    Retorna una nova cadena amb els índexs a 1 després d'aplicar el soroll.
    """
    ones = parse_sparse_fp(indices_str)

    # Vector booleà de longitud n_bits amb els bits actuals
    mask = np.zeros(n_bits, dtype=bool)
    for idx in ones:
        if 0 <= idx < n_bits:
            mask[idx] = True

    p0 = noise_0 / 100.0
    p1 = noise_1 / 100.0

    # Generem, per a cada bit, si es fliparà o no en funció del seu estat
    flip_zeros = rng.random(n_bits) < p0  # per als bits que són 0
    flip_ones  = rng.random(n_bits) < p1  # per als bits que són 1

    # Zeros que passen a 1
    mask[~mask & flip_zeros] = True
    # Uns que passen a 0
    mask[mask & flip_ones] = False

    new_indices = set(np.where(mask)[0])
    return fp_set_to_sparse_str(new_indices)

## 1. Lectura del fitxer

In [9]:
df = pd.read_csv(input_path)

if fp_in_col not in df.columns:
    raise ValueError(f"La columna '{fp_in_col}' no existeix al CSV.")

# Visualitzem el fitxer original
df.head(5)

,id,SMILES_input,fingerprints_input_ECFP4
0,1,OCCCCN1CCCC1,80 222 473 541 653 807 893 926 935 1028 1130 1...
1,2,N#CN(CC(N)=O)c1ccc(Cl)cc1-c1cccnn1,46 73 80 140 165 212 216 281 352 353 378 561 6...
2,3,Cc1cc[n+](C)cc1-c1ccccc1,222 263 350 352 389 463 482 673 691 736 1003 1...
3,4,COC(=O)NC1CCN(C(=O)CCCCC(=O)N2CCCCC2)C1,2 16 80 194 369 387 433 526 650 652 678 695 73...
4,5,COc1cccc(S(=O)(=O)NCC(=O)OC=CC2=CC=C2C(=O)Nc2c...,13 80 102 155 191 198 261 263 314 317 319 322 ...


## 2.Apliquem el soroll a la columna de fingerprints

In [10]:
df[fp_in_col] = df[fp_in_col].apply(
    lambda s: apply_noise_to_fp_sparse(s, n_bits=n_bits,
                                       noise_0=noise_0,
                                       noise_1=noise_1,
                                       rng=rng)
)

# Visualitzem el fitxer alterat
df.head(5)

,id,SMILES_input,fingerprints_input_ECFP4
0,1,OCCCCN1CCCC1,4 17 27 51 68 74 84 85 97 122 124 135 139 149 ...
1,2,N#CN(CC(N)=O)c1ccc(Cl)cc1-c1cccnn1,9 39 46 59 61 70 73 80 92 94 111 121 138 140 1...
2,3,Cc1cc[n+](C)cc1-c1ccccc1,23 48 55 57 69 101 106 120 128 130 140 157 202...
3,4,COC(=O)NC1CCN(C(=O)CCCCC(=O)N2CCCCC2)C1,2 9 16 46 72 80 85 88 105 106 110 129 148 149 ...
4,5,COc1cccc(S(=O)(=O)NCC(=O)OC=CC2=CC=C2C(=O)Nc2c...,10 45 70 79 80 102 107 139 143 144 171 175 178...


In [11]:
# Guardem amb el mateix nom de fitxer dins la carpeta de soroll
df.to_csv(output_path, index=False)